In [1]:
import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime, timedelta
import time

In [2]:
# today = datetime.today().strftime('%Y/%m/%d')
today = (datetime.today()-timedelta(days=1)).strftime('%Y/%m/%d')
url=f'https://pontofinal-macau.com/{today}/'
headers={'USER-AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36 Edg/128.0.0.0'}
r=requests.get(url, headers=headers)
time.sleep(10)
r

<Response [200]>

In [3]:
url

'https://pontofinal-macau.com/2026/03/22/'

In [4]:
soup=BeautifulSoup(r.content)
content=soup.find_all('div', class_='td-main-content-wrap td-container-wrap')[0]
posts=content.find_all('h3', class_='entry-title td-module-title')
valid_post=list(set([post.find_all('a')[0].get('href') for post in posts]))

In [5]:
url=valid_post[0]
headers={'USER-AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36 Edg/128.0.0.0'}
r=requests.get(url, headers=headers)
time.sleep(10)
soup=BeautifulSoup(r.content)

In [6]:
def get_article(url):
    headers={'USER-AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36 Edg/128.0.0.0'}
    r=requests.get(url, headers=headers)
    time.sleep(10)
    soup=BeautifulSoup(r.content)
    data=soup.find_all('div', {'class': 'td-pb-span8'})[-1]
    if data.find('time', {'class': 'entry-date updated td-module-date'}):
        date=data.find_all('time', {'class': 'entry-date updated td-module-date'})[0].get('datetime')
        date=datetime.strptime(date, '%Y-%m-%dT%H:%M:%S%z').strftime('%Y-%m-%d %H:%M:%S')
    else:
        date=None
    
    if soup.find('h1', class_='tdb-title-text'):
        title=soup.find_all('h1', class_='tdb-title-text')[0].get_text().strip()
    else:
        title=None

    if data.find('a', {'class': 'tdb-author-name'}):
        author=data.find_all('a', {'class': 'tdb-author-name'})[0].get_text().strip()
    else:
        author=None

    if data.find('div', class_='td-post-content'):
        content=data.find_all('div', class_='td-post-content')[0].get_text().strip()
    else:
        content=None

    if data.find('a', {'class': 'tdb-entry-category'}):
        cat=data.find_all('a', {'class': 'tdb-entry-category'})[0].get_text().strip()
    else:
        cat=None

    return date, title, cat, author, content

In [7]:
restructured_posts=[]
for idx, url in enumerate(valid_post):
    date, title, cat, author, content=get_article(url)
    dict_post={
        'title': title,
        'category': cat,
        'author': author,
        'link': url,
        'date': date,
        'content': content
    }
    restructured_posts.append(dict_post)

In [8]:

import xml.etree.ElementTree as ET

# Create the root of the RSS feed
rss = ET.Element('rss', version='2.0')
channel = ET.SubElement(rss, 'channel')

# Add channel elements
ET.SubElement(channel, 'title').text = 'Ponto_final'
ET.SubElement(channel, 'link').text = url
ET.SubElement(channel, 'description').text = 'Ponto_final RSS'

# Add items
for item in restructured_posts:
    item_elem = ET.SubElement(channel, 'item')
    ET.SubElement(item_elem, 'title').text = item['title']
    ET.SubElement(item_elem, 'link').text = item['link']
    ET.SubElement(item_elem, 'description').text = item['content']
    ET.SubElement(item_elem, 'pubDate').text = item['date']

# Convert to string and save to an XML file
rss_feed = ET.tostring(rss, encoding='utf-8', method='xml').decode()
with open('Ponto_final_pt.xml', 'w', encoding='utf-8') as xml_file:
    xml_file.write(rss_feed)

In [9]:
import base64

# Variables
token = 'Replaced by real toekn'
repo = 'mogcai/CPC_RSS'  # Replace with your GitHub username/repo
commit_message = 'Update RSS feed'
file_path='Ponto_final_pt.xml'

# Read the file content
with open(file_path, 'rb') as f:
    content = f.read()


In [10]:

# Encode the content
encoded_content = base64.b64encode(content).decode()
github_path='Ponto_final_pt.xml'
# Prepare the API endpoint URL
url = f'https://api.github.com/repos/{repo}/contents/{github_path}'



In [11]:

# Fetch the current SHA of the file (if it exists)
response = requests.get(url, headers=headers)
if response.status_code == 200:
    file_info = response.json()
    sha = file_info['sha']  # Get the SHA of the file
else:
    sha = None  # File does not exist
    print(f"Error fetching file info: {response.content.decode()}")


In [12]:

# Prepare the payload
payload = {
    'message': commit_message,
    'content': encoded_content,
    'branch': 'main',  # Change to 'master' if that is your default branch
}

if sha:
    payload['sha'] = sha  # Include the SHA if the file exists



# Make the API request to create or update the file
headers = {
    'Authorization': f'token {token}',
    'Accept': 'application/vnd.github.v3+json'
}


In [13]:
import json
response = requests.put(url, headers=headers, data=json.dumps(payload))

# Check the response
if response.status_code == 201:
    print('File uploaded successfully.')
elif response.status_code == 200:
    print('File updated successfully.')
else:
    print(f'Error: {response.content}')


File updated successfully.
